## ML_1044: Speculative Decoding

**Difficulty**: Hard | **TorchCode**: #34

### Problem
Implement speculative decoding token acceptance. For each draft token, accept with probability `min(1, p_target / p_draft)`. If rejected, resample from the adjusted distribution `max(0, p_target - p_draft)` (normalized). Return accepted tokens.

### Formula
$$P(\text{accept}_i) = \min\!\left(1,\, \frac{p_{\text{target}}(x_i)}{p_{\text{draft}}(x_i)}\right), \quad p_{\text{resample}} = \frac{\max(0,\, p_{\text{target}} - p_{\text{draft}})}{\sum_j \max(0,\, p_t - p_d)_j}$$

In [ ]:
import torch

def speculative_decode(target_probs, draft_probs, draft_tokens):
    K = len(draft_tokens)
    accepted = []
    for i in range(K):
        t = draft_tokens[i].item()
        ratio = target_probs[i, t] / max(draft_probs[i, t].item(), 1e-10)
        if torch.rand(1).item() < min(1.0, ratio.item()):
            accepted.append(t)
        else:
            adjusted = torch.clamp(target_probs[i] - draft_probs[i], min=0)
            s = adjusted.sum()
            if s > 0:
                adjusted = adjusted / s
            else:
                adjusted = torch.ones_like(adjusted) / adjusted.shape[0]
            accepted.append(torch.multinomial(adjusted, 1).item())
            return accepted
    return accepted

In [ ]:
import torch

# --- Test 1: Perfect draft — all accepted ---
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
accepted = speculative_decode(probs, probs, tokens)
assert len(accepted) == 4
for i in range(4):
    assert accepted[i] == tokens[i].item()

# --- Test 2: Output length bounded ---
torch.manual_seed(0)
K = 5
target = torch.softmax(torch.randn(K, 8), dim=-1)
draft = torch.softmax(torch.randn(K, 8), dim=-1)
tokens = torch.randint(0, 8, (K,))
accepted = speculative_decode(target, draft, tokens)
assert 1 <= len(accepted) <= K

# --- Test 3: All tokens valid ---
V = 8
for seed in range(20):
    torch.manual_seed(seed)
    target = torch.softmax(torch.randn(3, V), dim=-1)
    draft = torch.softmax(torch.randn(3, V), dim=-1)
    tokens = torch.randint(0, V, (3,))
    for t in speculative_decode(target, draft, tokens):
        assert 0 <= t < V

print('All tests passed!')